![](/Workspace/Users/sunnygupta2508@gmail.com/Databricks-Certified-Data-Engineer-Pro/Includes/images/bronze.png)

In [0]:
%run ../Includes/Copy-Datasets

In [0]:
files = dbutils.fs.ls(f"{bookstore.dataset_path}/kafka-raw")
display(files)

In [0]:
df_raw = (spark.read.json(f"{bookstore.dataset_path}/kafka-raw"))
display(df_raw)

In [0]:
from pyspark.sql import functions as F

def process_bronze():
    schema = "key binary, value binary, topic string, partition long, offset long, timestamp long"
    
    query = (
        spark.readStream
                .format("cloudFiles")
                .option("cloudFiles.format", "json")
                .schema(schema)
                .load(f"{bookstore.dataset_path}/kafka-raw")
                .withColumn("input_file_name", F.col("_metadata.file_path"))
                .withColumn("timestamp", (F.col("timestamp")/1000).cast("timestamp"))
                .withColumn("year_month", F.date_format("timestamp","yyyy-MM"))
            .writeStream
                .option("checkpointLocation", f"{bookstore.checkpoint_path}/bronze")
                .option("mergedSchema", True)
                .partitionBy("topic", "year_month")
                .trigger(availableNow=True)
                .table("bronze")
    )
    query.awaitTermination()

In [0]:
process_bronze()

In [0]:
df_bronze = spark.table("bronze")
display(df_bronze)

In [0]:
display(df_bronze.select("topic").distinct())

In [0]:
display(df_bronze.groupBy("topic").count())

In [0]:
bookstore.load_new_data()

In [0]:
process_bronze()

In [0]:
display(df_bronze.groupBy("topic").count())